In [1]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json


In [2]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [3]:
def decompress_zst_to_text(input_file, vocab):
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, extracts the "text" value from each line.
    
    Parameters:
        input_file_path (str): Path to the .zst file.
        vocab (set, optional): The vocabulary set to update with characters.
    
    Yields:
        str: The extracted text from each JSON line.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration
                # Process each line
                for line in lines:
                    if not line.strip():
                        continue
                    try:
                        data = json.loads(line)
                        text = data.get("text", "")
                        if vocab is not None:
                            vocab.update(set(text))
                        yield text.strip()
                    except json.JSONDecodeError as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")


In [4]:
folder_path = "openwebtext2"
output_file_train = "output_train_v3.txt"
output_file_val = "output_val_v3.txt"
vocab_file = "vocab_v3.txt"


In [5]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)

Total files: 179
['2005-06.jsonl.zst', '2005-07.jsonl.zst', '2005-08.jsonl.zst', '2005-09.jsonl.zst', '2005-10.jsonl.zst', '2005-11.jsonl.zst', '2005-12.jsonl.zst', '2006-01.jsonl.zst', '2006-02.jsonl.zst', '2006-03.jsonl.zst', '2006-04.jsonl.zst', '2006-05.jsonl.zst', '2006-06.jsonl.zst', '2006-07.jsonl.zst', '2006-08.jsonl.zst', '2006-09.jsonl.zst', '2006-10.jsonl.zst', '2006-11.jsonl.zst', '2006-12.jsonl.zst', '2007-01.jsonl.zst', '2007-02.jsonl.zst', '2007-03.jsonl.zst', '2007-04.jsonl.zst', '2007-05.jsonl.zst', '2007-06.jsonl.zst', '2007-07.jsonl.zst', '2007-08.jsonl.zst', '2007-09.jsonl.zst', '2007-10.jsonl.zst', '2007-11.jsonl.zst', '2007-12.jsonl.zst', '2008-01.jsonl.zst', '2008-02.jsonl.zst', '2008-03.jsonl.zst', '2008-04.jsonl.zst', '2008-05.jsonl.zst', '2008-06.jsonl.zst', '2008-07.jsonl.zst', '2008-08.jsonl.zst', '2008-09.jsonl.zst', '2008-10.jsonl.zst', '2008-11.jsonl.zst', '2008-12.jsonl.zst', '2009-01.jsonl.zst', '2009-02.jsonl.zst', '2009-03.jsonl.zst', '2009-04.jsonl.z

In [6]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

['2018-02.jsonl.zst', '2009-11.jsonl.zst', '2006-09.jsonl.zst', '2008-12.jsonl.zst', '2006-07.jsonl.zst', '2008-06.jsonl.zst', '2010-06.jsonl.zst', '2015-08.jsonl.zst', '2010-07.jsonl.zst', '2007-01.jsonl.zst', '2010-11.jsonl.zst', '2007-12.jsonl.zst', '2018-05.jsonl.zst', '2005-08.jsonl.zst', '2016-08.jsonl.zst', '2016-09.jsonl.zst', '2015-06.jsonl.zst', '2018-09.jsonl.zst', '2019-07.jsonl.zst', '2010-12.jsonl.zst', '2013-04.jsonl.zst', '2019-11.jsonl.zst', '2007-07.jsonl.zst', '2016-06.jsonl.zst', '2017-10.jsonl.zst', '2018-08.jsonl.zst', '2019-12.jsonl.zst', '2011-08.jsonl.zst', '2017-03.jsonl.zst', '2017-07.jsonl.zst', '2006-08.jsonl.zst', '2009-10.jsonl.zst', '2016-02.jsonl.zst', '2014-02.jsonl.zst', '2009-02.jsonl.zst', '2015-09.jsonl.zst', '2011-03.jsonl.zst', '2012-02.jsonl.zst', '2018-10.jsonl.zst', '2005-06.jsonl.zst', '2015-01.jsonl.zst', '2006-10.jsonl.zst', '2016-10.jsonl.zst', '2015-11.jsonl.zst', '2013-10.jsonl.zst', '2010-10.jsonl.zst', '2018-06.jsonl.zst', '2012-05.jso

In [7]:
# Split files into train/val (90%/10%)
split_index = int(total_files * 0.9)
files_train = files[:split_index]
files_val = files[split_index:]
vocab = set()

In [10]:
# Process training files
with open(output_file_train, "w", encoding="utf-8") as outf:
    for filename in tqdm(files_train, total=len(files_train), desc="Processing Train"):
        file_path = os.path.join(folder_path, filename)
        try:
            for text_line in decompress_zst_to_text(file_path, vocab):
                outf.write(text_line.strip() + "\n")  # Write only the text line
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

Processing Train: 100%|██████████| 161/161 [24:20<00:00,  9.07s/it]


In [11]:
# Process validation files
with open(output_file_val, "w", encoding="utf-8") as outf:
    for filename in tqdm(files_val, total=len(files_val), desc="Processing Val"):
        file_path = os.path.join(folder_path, filename)
        try:
            for text_line in decompress_zst_to_text(file_path, vocab):
                outf.write(text_line.strip() + "\n")  # Write only the text line
        except Exception as e:
            print(f"Error processing {file_path}: {e}")


Processing Val: 100%|██████████| 18/18 [01:36<00:00,  5.36s/it]


In [8]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from tokenizers.processors import ByteLevel  # import the processor
from transformers import GPT2TokenizerFast

def build_custom_bpe_tokenizer_from_files(file_paths, vocab_size=30522):

    # set token for out-of-vocabulary words
    print("set token for out-of-vocabulary words")
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    

    # Configure pre/post processors
    print("Configure pre/post processors")
    # Step 1: Pre-tokenizer (split bytes)
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        # 1.1. Split punctuation into separate tokens
        # pre_tokenizers.Punctuation(),
        # 1.2. Apply Byte Level processing
        pre_tokenizers.ByteLevel(add_prefix_space=True)
    ])
    # Step 2: Post-processor (remove control bytes during decoding)
    tokenizer.post_processor = ByteLevel()  # Use the processor directly without eot_token  


    # specify training strategy for the BPE tokenizer
    print("specify training strategy for the BPE tokenizer")
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
    )
    
    print("train tokenizer")
    tokenizer.train(files=file_paths, trainer=trainer)
    
    # Save to temporary JSON (intermediate step)
    print("Save custom_tokenizer to temporary JSON")
    tokenizer.save("custom_tokenizer.json")
    
    # Convert to HuggingFace compatible tokenizer and save to a DIRECTORY
    print("Convert to HuggingFace compatible tokenizer and save to a DIRECTORY")
    hf_tokenizer = GPT2TokenizerFast(tokenizer_file="custom_tokenizer.json")
    hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'})  # Explicitly add [PAD] token
    hf_tokenizer.save_pretrained("custom_bpe_tokenizer")  # Save to directory
    
    return hf_tokenizer
        

In [9]:
# Train the tokenizer on your data
trained_tokenizer = build_custom_bpe_tokenizer_from_files([output_file_train, output_file_val], vocab_size=30522) 

set token for out-of-vocabulary words
Configure pre/post processors
specify training strategy for the BPE tokenizer
train tokenizer
Save custom_tokenizer to temporary JSON
Convert to HuggingFace compatible tokenizer and save to a DIRECTORY


In [10]:
# Load the saved tokenizer
loaded_tokenizer = GPT2TokenizerFast.from_pretrained(
    "custom_bpe_tokenizer",  # directory name, NOT the JSON file
    model_max_length=1024
)

# Double-check and set padding token (insurance)
if not loaded_tokenizer.pad_token:
    loaded_tokenizer.add_special_tokens({'pad_token': '[PAD]'})

vocab_size = loaded_tokenizer.vocab_size  # e.g., 30,522

In [14]:

# Example usage
text = "Hello, world! I like apple juice - I drink it every day. Isn't that too much?"
encoded = loaded_tokenizer(text, return_tensors='pt', padding=True, truncation=True)
print(encoded)

# Output tokens
decoded = loaded_tokenizer.decode(
    encoded['input_ids'][0], 
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
    ).replace("Ġ","")
print(decoded)

{'input_ids': tensor([[17668,    16,   923,     5,   276,   551, 21384, 17318,   619,   276,
          4886,   307,   769,  1107,    18, 25722,   714,   289,  1205,   857,
            35]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Hello, world! I like apple juice - I drink it every day. Isn 't that too much?


In [12]:
import cProfile

def test_tokenizer():
    encoded = loaded_tokenizer.encode("Hello, world!" * 128)

cProfile.run('test_tokenizer()')

         69 function calls in 0.002 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.002    0.002 873259721.py:3(test_tokenizer)
        1    0.000    0.000    0.002    0.002 <string>:1(<module>)
        2    0.000    0.000    0.000    0.000 __init__.py:1091(__init__)
        4    0.000    0.000    0.000    0.000 __init__.py:1108(__setitem__)
        2    0.000    0.000    0.000    0.000 _collections_abc.py:986(update)
        2    0.000    0.000    0.000    0.000 abc.py:117(__instancecheck__)
        1    0.000    0.000    0.002    0.002 tokenization_gpt2_fast.py:109(_batch_encode_plus)
        1    0.000    0.000    0.002    0.002 tokenization_gpt2_fast.py:118(_encode_plus)
        2    0.000    0.000    0.000    0.000 tokenization_utils_base.py:221(__init__)
        2    0.000    0.000    0.000    0.000 tokenization_utils_base.py:260(__getitem__)
        1    0.000    0.000    0.002    0.0